## **Сбор данных и preprocessing data**

### Я выбрал два датасета:
- **Russian Language Toxic Comments** (https://www.kaggle.com/datasets/blackmoon/russian-language-toxic-comments)
- **Mnwa/russian-toxic** (https://huggingface.co/datasets/Mnwa/russian-toxic?library=pandas)
### В итоге у меня получилось около 329 тыс. строк в датасете.

In [15]:
import torch
torch.cuda.empty_cache()
import random
import numpy as np
from pathlib import Path

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(42)

In [16]:
DATA_DIR = Path("data")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE # нужен, чтобы в torch выполнялись быстрее вычисления, направляет вычисления на GPU

'cuda'

In [17]:
import pandas as pd

df_1 = pd.read_csv("data/labeled.csv")
df_1.columns = ["text", "toxic"]
print(df_1.head())

                                                text  toxic
0               Верблюдов-то за что? Дебилы, бл...\n    1.0
1  Хохлы, это отдушина затюканого россиянина, мол...    1.0
2                          Собаке - собачья смерть\n    1.0
3  Страницу обнови, дебил. Это тоже не оскорблени...    1.0
4  тебя не убедил 6-страничный пдф в том, что Скр...    1.0


In [ ]:
from datasets import load_dataset
from huggingface_hub import login

login("") # заходим в профиль, чтобы подгрузить датасет

ds = load_dataset("Mnwa/russian-toxic") # загружаем датасет
df_2 = pd.concat([ds["train"].to_pandas(), ds["test"].to_pandas()], ignore_index=True) # объединяем тестовую и тренировочную выборку
df_2.columns = ["text", "toxic"]
df_2["toxic"] = 1 - df_2["toxic"]
print(df_2.head())

                                                text  toxic
0  сильный лидер не врет, сильный лидер не пидр, ...      1
1  на исходном фото надпись: вот так наша бабуля ...      0
2  ► Получайте заказы &quot;Мастер на час&quot; в...      0
3      а вдруг и свершится чудо ! а какая моя душа ?      0
4  [id731232234|Count], я бы и ленту закрыл, чтоб...      0


### Объединим эти два датасета и почистим их от дубликатов, html текста и т.п.

In [19]:
df = pd.concat([df_1, df_2], ignore_index=True)
print(df.head())

                                                text  toxic
0               Верблюдов-то за что? Дебилы, бл...\n    1.0
1  Хохлы, это отдушина затюканого россиянина, мол...    1.0
2                          Собаке - собачья смерть\n    1.0
3  Страницу обнови, дебил. Это тоже не оскорблени...    1.0
4  тебя не убедил 6-страничный пдф в том, что Скр...    1.0


In [20]:
df = df.drop_duplicates()
df["toxic"].value_counts()


,count
toxic,
0.0,255012
1.0,70026


### Для того чтобы этот дизбаланс классов не сыграл против нас, нужно увеличить вес для токсичного сообщения, чтобы общая ошибка была больше. Я взял такую ошибку:
- **neg_count/pos_count * 1.8**

In [21]:
neg_count, pos_count = df["toxic"].value_counts().values
pos_weight = torch.tensor([neg_count/pos_count * 1.8]).to(DEVICE) # вычисляем вес для 1 (токсичного)
print(pos_weight)

tensor([6.5550], device='cuda:0', dtype=torch.float64)


### Преобразование текста:

In [22]:
import re
import html # для преобразования htlml символов в нормальные
from bs4 import BeautifulSoup # удаляет разметку html

# убираем исключение
from bs4 import MarkupResemblesLocatorWarning
import warnings

warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning) # предупреждение от разработчика о том, правильно ли мы используем библиотеку


def clean_text(text: str) -> str:
    text = html.unescape(text) # преобразует символы html в обычные символы(из "Tom &amp; Jerry" в "Tom & Jerry")
    text = BeautifulSoup(text, "html.parser").get_text() # Удаляет разметку html текста (Пример: из "<div>Hello <b>world</b></div>" в "Hello world")
    text = re.sub(r"\s+", " ", text) # убрать лишние пробел
    text = re.sub(r"[^а-яёa-z0-9\s]", " ", text, flags=re.IGNORECASE) # флаг нужен, чтобы большие буквы тоже не трогал
    return text.strip().lower() # strip - удаляет все пробелы справа и слева, lower() - все буквы к нижнему регистру


df["text"] = df["text"].apply(clean_text)
print(df.head())

                                                text  toxic
0                    верблюдов то за что  дебилы  бл    1.0
1  хохлы  это отдушина затюканого россиянина  мол...    1.0
2                            собаке   собачья смерть    1.0
3  страницу обнови  дебил  это тоже не оскорблени...    1.0
4  тебя не убедил 6 страничный пдф в том  что скр...    1.0


### Подготовим наш датасет для обучения:

In [23]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.3,
    stratify=df["toxic"],
    shuffle=True,
    random_state=42
    )

val_df, test_df = train_test_split(
    temp_df,
    test_size=1/3,
    stratify=temp_df["toxic"],
    shuffle=True,
    random_state=42
    )



In [24]:
import pickle
from typing import List, Dict
from vocab import Vocab, tokenize


# делаем словарь из уникальных слов
vocab = Vocab(list(train_df["text"].head(50000).map(tokenize)))
print(len(vocab))
o
# Сохраняем словарь для дальнейшего использования в тг боте
with open("data/vocab.pkl", "wb") as f:
    pickle.dump(vocab, f)

140604


In [25]:
from torch.utils.data import Dataset, DataLoader

# датасет для тренировки
class CensorDataset(Dataset):
    def __init__(self, df, vocab: Vocab) -> None:
        self.df = df.reset_index(drop=True)
        self.vocab = vocab

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        row = self.df.iloc[idx] # возвращает строчку с индексами idx из столбца text

        tokens = tokenize(row["text"])
        ids = self.vocab.encode(tokens)

        return {
            "text": torch.tensor(ids, dtype=torch.long),
            "toxic": torch.tensor(row["toxic"], dtype=torch.float)
        }


# функция, приводящая все предложения к одному размеру
def collate_fn(batch: List[Dict[str, torch.Tensor]], pad_id: int) -> Dict[str, torch.Tensor]:
    ids = [x["text"] for x in batch]

    toxic = torch.tensor(
        [x["toxic"] for x in batch],
        dtype=torch.float
    )

    max_len = max(len(x) for x in ids)

    padded = torch.full(
        (len(batch), max_len),
        pad_id,
        dtype=torch.long
    ) # создаем torch из нулей с размером (размер батча, максимальный размер предложения в батче)


    # заполняем батч предложениями
    for i, seq in enumerate(ids):
        padded[i, :len(seq)] = seq


    return {
        "text": padded,
        "toxic": toxic
    }

In [26]:
train_ds = CensorDataset(train_df, vocab)
val_ds = CensorDataset(val_df, vocab)
test_ds = CensorDataset(test_df, vocab)


train_loader = DataLoader(
    train_ds,
    batch_size=128,
    shuffle=True,
    collate_fn=lambda batch: collate_fn(batch, vocab.pad_id)
)

val_loader = DataLoader(
    val_ds,
    batch_size=128,
    shuffle=True,
    collate_fn=lambda batch: collate_fn(batch, vocab.pad_id)
)

test_loader = DataLoader(
    test_ds,
    batch_size = 128,
    shuffle=True,
    collate_fn=lambda batch: collate_fn(batch, vocab.pad_id)
)

batch = next(iter(train_loader))
print(batch["text"].shape)
print(batch["toxic"].shape)

torch.cuda.empty_cache()

torch.Size([128, 323])
torch.Size([128])


### Построим функции для обучения модели:

In [27]:
from sklearn.metrics import confusion_matrix
from model import ToxicCNNBiLSTM

def train_epoch(model: ToxicCNNBiLSTM,
                dataloader: torch.utils.data.DataLoader,
                criterion: torch.nn.Module,
                optimizer: torch.optim.Optimizer,
                device: str
                ) -> float:

    model.train()

    total_loss = 0

    for batch in dataloader:

        x_batch = batch["text"].to(device)
        y_batch = batch["toxic"].to(device).float()

        logits = model(x_batch)

        loss = criterion(logits.squeeze(1), y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)



def validate_epoch(model: ToxicCNNBiLSTM,
                   val_loader: torch.utils.data.DataLoader,
                   criterion: torch.nn.Module,
                   device: str,
                   threshold: int=0.5
                   ) -> Dict[str, float]:
    model.eval()
    total_loss = 0
    all_preds = []
    all_toxic = []

    with torch.no_grad():
        for batch in val_loader:
            x_batch = batch["text"].to(device)
            y_batch = batch["toxic"].to(device).float()

            logits = model(x_batch).squeeze(1)
            loss = criterion(logits, y_batch)
            total_loss += loss.item()

            preds = (logits.sigmoid() > threshold).cpu().numpy() # сначала torch.tensor отправляем на cpu и полсе преобразуем в numpy
            all_preds.extend(preds)
            all_toxic.extend(y_batch.cpu().numpy()) # также как и с preds

    tn, fp, fn, tp = confusion_matrix(all_toxic, all_preds).ravel() # ravel - возвращает одномерный массив вместо матрицы

    return {
        "loss": total_loss / len(val_loader),
        "accuracy": (tp + tn) / (tp + tn + fp + fn),
        "recall": tp/(tp + fn),
        "precision": tp/(tp + fp),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }


### Теперь собственно обучим нашу модель на наших данных:

In [28]:

model = ToxicCNNBiLSTM(
    vocab_size=len(vocab),
    embedding_dim=128,
    hidden_dim=128,
    num_filters=100,
    kernel_size=3,
    dropout=0.3,
    padding=vocab.pad_id
).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight) # pos_weight - чтобы увеличить вес ошибки при False Negative

n_epochs = 5
best_acc = 0
best_model = None

for epoch in range(n_epochs):
    train_loss = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
    result = validate_epoch(model, val_loader, criterion, DEVICE, 0.7)
    print(f"Эпоха {epoch}: train_loss={train_loss:.4f}, val_loss={result['loss']:.4f}, val_acc={result['accuracy']:.4f}")
    print(f"recall = {result['recall']}, precision = {result['precision']}")
    print(f"fn = {result['fn']}, tn = {result['tn']}, fp = {result['fp']}, tp = {result['tp']}")

    if result["accuracy"] > best_acc:
      best_acc = result["accuracy"]
      torch.save(model.state_dict(), "data/best_model.pt") # сохраняем лучшую модель чтобы после ею пльзоваться в самом тг боте


print(f"ОБУЧЕНИЕ ЗАКОНЧЕНО! Лучшая accuracy: {best_acc:.4f}")


Эпоха 0: train_loss=0.8080, val_loss=0.5672, val_acc=0.9271
recall = 0.8044983934309176, precision = 0.8490580256217031
fn = 2738, tn = 49000, fp = 2003, tp = 11267
Эпоха 1: train_loss=0.4230, val_loss=0.4613, val_acc=0.9330
recall = 0.8766868975365941, precision = 0.8236399007177836
fn = 1727, tn = 48374, fp = 2629, tp = 12278
Эпоха 2: train_loss=0.2856, val_loss=0.4811, val_acc=0.9354
recall = 0.8808996786861835, precision = 0.8296570275722932
fn = 1668, tn = 48470, fp = 2533, tp = 12337
Эпоха 3: train_loss=0.2047, val_loss=0.5449, val_acc=0.9375
recall = 0.8872545519457337, precision = 0.8332327499497083
fn = 1579, tn = 48516, fp = 2487, tp = 12426
Эпоха 4: train_loss=0.1520, val_loss=0.6465, val_acc=0.9343
recall = 0.8950374866119243, precision = 0.8171979920464176
fn = 1470, tn = 48199, fp = 2804, tp = 12535
ОБУЧЕНИЕ ЗАКОНЧЕНО! Лучшая accuracy: 0.9375


### Посмотрим, что покажет наша модель на тестовых данных:

In [29]:
model.load_state_dict(torch.load("data/best_model.pt", map_location=DEVICE)) # подгружает все на устройство, на котором выполнялось
test_result = validate_epoch(model, val_loader, criterion, DEVICE, 0.9)
print(f"test_loss: {test_result['loss']}, test_accuracy: {test_result['accuracy']}")
print(f"test_recall: {test_result['recall']}, test_precision: {test_result['precision']}")


test_loss: 0.5445533791102293, test_accuracy: 0.947744892936254
test_recall: 0.8446269189575152, test_precision: 0.9064367816091954


### **Вывод:**
Модель показала очень хорошие результаты на тестовых данных при threshold = 0.9:

| **Метрика** | **Значение** |
|-------------------|------------------|
| **Recall**    | 0.84    |
| **Precision**    |  0.90   |
| **Accuracy**  |  0.95  |